## Visualize canon threads

Group the indexed `result/sample-data.json` hits by `thread_id` / `canon_order`.
`_id` suffix encodes dedup: no suffix = original, `m` = near-duplicate (Levenshtein > 0.9),
`mm` = exact `content_hash` duplicate. Each duplicate hangs off the slot it collapsed into.


In [ ]:
!pip install requests

In [4]:
import re
from collections import defaultdict
import requests

resp = requests.post(
    "http://my.local/es/email/_search",
    json={"size": 1000, "sort": [{"canon_order": "asc"}]},
)
resp.raise_for_status()
hits = [h["_source"] for h in resp.json()["hits"]["hits"]]


def variant(es_id):
    m = re.search(r"m+$", es_id)
    return m.group(0) if m else ""


threads = defaultdict(lambda: defaultdict(list))
for s in hits:
    threads[s["thread_id"]][s["canon_order"]].append(s)

for tid, slots in threads.items():
    subj = next(iter(slots.values()))[0]["subject"]
    print(f"THREAD {tid[:8]}  {subj}")
    for order in sorted(slots):
        rows = sorted(slots[order], key=lambda s: len(variant(s["id"])))
        for i, s in enumerate(rows):
            v = variant(s["id"])
            tag = "orig" if not v else ("=dup" if len(v) > 1 else "~dup")
            branch = f"  [{order}]" if i == 0 else "       +"
            who = ",".join(s["from"])
            print(f"{branch} {tag:4} {s['id']:11} {who:18} {s['content']}")
    print()

THREAD d1de003c  Q2 budget review
  [0] orig doc10_0     dave@example.com   Hi Erin, Can you pull together the Q2 budget numbers before our review? Thanks. Dave
  [1] orig doc10_1     erin@example.com   Hi Dave, Sure, I'll have the Q2 figures ready by Thursday. Erin
       + ~dup doc10_1m    erin@example.com   Hi Dave, Sure, I will have the Q2 figures ready by Thursday. Erin
  [2] orig doc10_2     dave@example.com   Great, let's meet Friday at 2 pm to go over them. Dave
       + ~dup doc10_2m    dave@example.com   Great, let's meet Friday at 2pm to go over them. Dave

THREAD a2a9f2a4  Project kickoff
  [0] orig doc1_0      alice@example.com  Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice
  [1] orig doc2_1      bob@example.com    Hi Alice, Tuesday works great for me. Let's say 10am? Bob
  [2] orig doc4_2      alice@example.com  Perfect, 10am Tuesday it is. I'll send a calendar invite. Alice
       + ~dup doc4_2m     alice@example.com  Perfect,